### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following: 
- Tracking agent behavior with logging, analytics and debugging
- Transforming prompts, tool selection, and output formatting
- Adding retries, fallbacks, and early termination logic
- Applying rate limits, guardrails and PII detection

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running converstions that exceed context window
- Multi-turn dialogues wiht extensive history
- Applicaitions where preserving full conversation context matters

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased Summarization
agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
) 

In [ ]:
### Run with thread id

config = {"configurable":{"thread_id":"test-2"}}

In [ ]:
## Test data
questions =[
    "What is 2+2",
    "What is 5*8",
    "What is 15-7",
    "What is 100/4",
    "What is 19*4",
    "What is 234-4",
    "What is 23+4"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages:{len(response['messages'])}")


### Trigger based on token size

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

@tool
def search_hotel(city:str)->str:
    """Search hotels - returns a long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. Holiday Inn - 3 star, $100/night, wifi, gym
    3. Marriot - 5 star, $330/night, spa, pool, gym

"""

agent=create_agent(
    model="gpt-4o-mini",
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens",550),
            keep=("tokens",200)
        ),
        ],
)

config={"configurable":{"thread_id":"user124"}}

def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars/4 # 4 chars= 1 token

In [ ]:
# Run Test
cities = ["Islamabad", "Lohore", "Karachi", "Paris", "New York", "Jeddah"]

for city in cities:
    response=agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]}, config=config
        )
    tokens=count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

### Summarization based on Fraction

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

@tool
def search_hotel(city:str)->str:
    """Search hotels - returns a long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. Holiday Inn - 3 star, $100/night, wifi, gym
    3. Marriot - 5 star, $330/night, spa, pool, gym

"""

agent=create_agent(
    model="gpt-4o-mini",
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("fraction",0.005), # 0.5% = ~640 tokens. it is based on the context window of the LLM
            keep=("fraction",0.002),    # 0.2% = ~256 tokens
        ),
        ],
)

config={"configurable":{"thread_id":"user124"}}

def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars/4 # 4 chars= 1 token


cities = ["Islamabad", "Lohore", "Karachi", "Paris", "New York", "Jeddah"]

for city in cities:
    response=agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]}, config=config
        )
    tokens=count_tokens(response["messages"])
    fraction=tokens/128000 # gpt-40-mini context
    print(f"{city}: ~{tokens} tokens({fraction:0.4%}, {len(response['messages'])} msgs")
    print(f"{(response['messages'])}")

### Human in the Loop (HITL) Middleware

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. HITL is useful for the following:

- High stakes operations requiring human approval(e.g. database writes, financial transactions)
- Compliance workflows where human oversight is mandatory
- Long-runnig conversations where human feedback guides the agent

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id:str) -> str:
    """Mock function to read the email by its ID"""
    return f"Email content for ID:{email_id}"


def send_email_tool(recipient:str, subject:str, body:str) -> str:
    """Mock function to send an email"""
    return f"Email sent to ID: {recipient} with subject{subject}"

In [ ]:
agent = create_agent(
    model="gpt-4o-mini",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [ ]:
config = {"configurable":{"thread_id":"test_approve1"}}

# Step 1: Request
result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to ahmed@example.com with subject 'Hello' and body 'How are you'")]},
    config=config
)


### Reject

In [ ]:
from langgraph.types import Command

config = {"configurable":{"thread_id":"test_reject"}}

# Step 1: Request
result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to ahmed@example.com with subject 'Hello' and body 'How are you'")]},
    config=config
)
# Step 2: Approve
if "__interrupt__" in result:
    print(" Paused! Approving ...")

    result = agent.invoke(
        Command (resume = {"decisions": [{"type":"reject"}
                                         ]
                          }
                ), config=config 
    )

print(f" Result: {result['messages'][-1].content}")

### Edit

In [ ]:
from langgraph.types import Command

config = {"configurable":{"thread_id":"test_edit3"}}

# Step 1: Request
result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to wrong@example.com with subject 'Hello' and body 'How are you'")]},
    config=config
)
# Step 2: Approve
if "__interrupt__" in result:
    print(" Paused! Approving ...")

    result = agent.invoke(
        Command(
            resume ={
                "decisions": [
                    {
                        "type":"edit",
                        "edited_action":{
                            "name":"send_email_tool",
                            "args":{
                                "recipient":"Ahmed@example.com",
                                "subject":" Hello ",
                                "body": " Edited: How are you"
                            }
                        }
                    }
                ]
            }
        ), config=config 
    )

print(f" Result: {result['messages'][-1].content}")

In [ ]:
result

### With Human input

In [3]:
from langgraph.types import Command

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

def read_email_tool(email_id:str) -> str:
    """Mock function to read the email by its ID"""
    return f"Email content for ID:{email_id}"

def send_email_tool(recipient:str, subject:str, body:str) -> str:
    """Mock function to send an email"""
    return f"Email sent to ID: {recipient} with subject{subject}"

# Create Agent
agent = create_agent(
    model="gpt-4o-mini",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

config = {"configurable":{"thread_id":"test_HITL"}}

# Step 1: Request
result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to wrong@example.com with subject 'Hello' and body 'How are you'")]},
    config=config
)

#Step 2: HITL
if "__interrupt__" in result:
    print("Paused for human approval.")

    human_decision = input("approve, edit, or reject? ").strip().lower()

    if human_decision == "approve":
        resume_data = {
            "decisions": [{"type": "approve"}]
        }

    elif human_decision == "reject":
        resume_data = {
            "decisions": [{"type": "reject"}]
        }

    elif human_decision == "edit":
        recipient = input("New recipient: ")
        subject = input("New subject: ")
        body = input("New body: ")

        resume_data = {
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email_tool",
                        "args": {
                            "recipient": recipient,
                            "subject": subject,
                            "body": body,
                        },
                    },
                }
            ]
        }

    result = agent.invoke(
        Command(resume=resume_data),
        config=config,
    )

print(result["messages"][-1].content)

Paused for human approval.
The email has been sent to **ahmed@example.com** with the subject "Hello" and the body "How are you?" If you need any further assistance or want to send another email, just let me know!
